In [1]:
from sqlalchemy import create_engine, URL
import os
from dotenv import load_dotenv
from pathlib import Path

load_dotenv()
url = URL.create(
    drivername="postgresql",
    username=os.environ.get("POSTGRES_USER"),
    password=Path("/run/secrets/postgres_password").read_text().strip(),
    host=os.environ.get("POSTGRES_HOST"),
    port=os.environ.get("POSTGRES_PORT"),
    database=os.environ.get("POSTGRES_DB")
)

engine = create_engine(url)

In [2]:
from sqlalchemy import text

def drop_table():
  query = text("""
  DROP TABLE IF EXISTS employees;
  """)

  with engine.begin() as conn:
    conn.execute(query)

In [3]:
def create_table():
  query = text("""
    CREATE TABLE IF NOT EXISTS employees (
      emp_id INT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
      first_name VARCHAR(50) NOT NULL,
      last_name VARCHAR(50) NOT NULL,
      experience NUMERIC(4, 1) NOT NULL,
      skills TEXT NOT NULL
    );
  """)

  with engine.begin() as conn:
    conn.execute(query)

In [4]:
import pandas as pd

def populate_table():
  df = pd.read_csv("employees.csv")
  
  df.to_sql("employees", engine, if_exists= 'delete_rows', index=False)
  print("database loaded")

In [5]:
drop_table()
create_table()
populate_table()

database loaded
